<a href="https://colab.research.google.com/github/LivingstonTardzenyuy/Generative-AI/blob/main/working_with_custom_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import userdata

grok_key = userdata.get('GROK_api')

In [ ]:
!pip install langchain-groq
from langchain_groq import ChatGroq

chatModel = ChatGroq(
    model = "openai/gpt-oss-20b",
    groq_api_key = grok_key
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.5/471.5 kB 17.8 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.79
    Uninstalling langchain-core-0.3.79:
      Successfully uninstalled langchain-core-0.3.79
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.27 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.0.5 which is incompatible.


In [ ]:
messages = [
    ("system", "You are a historian expert in the Kennedy family."),
    ("user", "who is Kongnyuy Livingston ?"),
]

response = chatModel.invoke(messages)
response.content

'I’m not finding any historical record or reputable source that lists a person named **Kongnyuy Livingston**. It’s possible the name is misspelled, a nickname, or a fictional/obscure figure that hasn’t been documented in the usual genealogical or historical archives.\n\nIf you have any additional context—such as a time period, geographic location, or a related family name (e.g., a connection to the Kennedy family, the Livingston clan in New\u202fYork, or a particular document or publication)—that would help me narrow the search. Alternatively, if “Kongnyuy” is a typo, let me know the correct spelling and I’ll dig into it again.'

## From the above we see the limitations with our LLM.
it does not have access to many much information.

Now we will have to train it with our own custom data.

# Getting some data from drive and loading it.

In [ ]:
# Install gdown library if not already installed
!pip install gdown

In [ ]:
import os
import gdown

# The folder 'data' should already exist from previous steps
folder_name = 'data'
os.makedirs(folder_name, exist_ok=True)

# List of files to download
files_to_download = [
    {
        'url': 'https://docs.google.com/document/d/1GgsuYbVOMhFX485g9DfQCCga5F0wFupC2xye-C7J5QQ/edit?tab=t.0',
        'type': 'doc',
        'output_name': 'document1.txt'
    },
    {
        'url': 'https://docs.google.com/spreadsheets/d/1a-UcUgnd2YFqb8pj0bKMxN9FEk9qNO4qJKYTnoWjsdA/edit?gid=0#gid=0',
        'type': 'sheet',
        'output_name': 'spreadsheet1.csv'
    },
    {
        'url': 'https://docs.google.com/document/d/1EeLapuw8QVuVFSAkeCETGKOPi0yUT7PaJ0wgaF8hD7A/edit?tab=t.0#heading=h.mbra3ggubsrw',
        'type': 'doc',
        'output_name': 'document2.txt'
    }
]

for file_info in files_to_download:
    url = file_info['url']
    file_type = file_info['type']
    output_name = file_info['output_name']

    # Extract document/spreadsheet ID
    if 'document/d/' in url:
        document_id = url.split('document/d/')[1].split('/')[0]
    elif 'spreadsheets/d/' in url:
        document_id = url.split('spreadsheets/d/')[1].split('/')[0]
    else:
        print(f"Skipping {url}: Unrecognized Google Drive URL format.")
        continue

    # Construct export URL based on file type
    if file_type == 'doc':
        export_url = f'https://docs.google.com/document/d/{document_id}/export?format=txt'
    elif file_type == 'sheet':
        export_url = f'https://docs.google.com/spreadsheets/d/{document_id}/export?format=csv'
    else:
        print(f"Skipping {url}: Unsupported file type {file_type}.")
        continue

    # Define the full path for the downloaded file
    output_file_path = os.path.join(folder_name, output_name)

    print(f"\nDownloading {output_name} from {url}...")
    gdown.download(export_url, output_file_path, quiet=False)
    print(f"File downloaded to: {output_file_path}")

/usr/local/lib/python3.12/dist-packages/gdown/parse_url.py:48: UserWarning: You specified a Google Drive link that is not the correct link to download a file. You might want to try `--fuzzy` option or the following url: https://drive.google.com/uc?id=None
  warnings.warn(
Downloading...
From: https://docs.google.com/document/d/1GgsuYbVOMhFX485g9DfQCCga5F0wFupC2xye-C7J5QQ/export?format=txt
To: /content/data/document1.txt
7.48kB [00:00, 9.32MB/s]


File downloaded to: data/document1.txt



Downloading...
From: https://docs.google.com/spreadsheets/d/1a-UcUgnd2YFqb8pj0bKMxN9FEk9qNO4qJKYTnoWjsdA/export?format=csv
To: /content/data/spreadsheet1.csv
8.96kB [00:00, 6.23MB/s]


File downloaded to: data/spreadsheet1.csv



Downloading...
From: https://docs.google.com/document/d/1EeLapuw8QVuVFSAkeCETGKOPi0yUT7PaJ0wgaF8hD7A/export?format=txt
To: /content/data/document2.txt
8.95kB [00:00, 9.49MB/s]

File downloaded to: data/document2.txt


You can verify the downloaded files by listing the contents of the `data` folder:

In [ ]:
import os

folder_name = 'data'
print(f"Contents of '{folder_name}' folder:")
for filename in os.listdir(folder_name):
    print(filename)

Contents of 'data' folder:
document1.txt
spreadsheet1.csv
document2.txt


# Data Loader

In [ ]:
!pip install --upgrade langchain-community langchain-groq langchain-core langchain

  Using cached langchain_core-1.0.5-py3-none-any.whl.metadata (3.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.1 MB/s eta 0:00:00
Using cached langchain_core-1.0.5-py3-none-any.whl (471 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.7/93.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.2/410.2 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.3/208.3 kB 19.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32

In [ ]:
# txt data loading.

from langchain_community.document_loaders import TextLoader

loader1 = TextLoader("data/document1.txt")
loader2 = TextLoader("data/document2.txt")


# Load the data.
documents1 = loader1.load()
documents2 = loader2.load()

In [ ]:
documents1

[Document(metadata={'source': 'data/document1.txt'}, page_content='\ufeffCAITCC Vocational Training Program\n\n\nModule: Introduction to Prompt Engineering\n\n\nPrepared by: Kongnyuy Livingston & Nyuydini Bill\n\n\nFor: CAITCC Vocational Training Center\n\n\nPrograms: AI Product Manager | AI Product Marketing Manager | AI Engineer\nPhase 1: Understanding the Basics of Programming (Using Python)\n(For CAITCC Vocational Training – AI Product Management Program)\n________________\n\n\n1. Introduction to Programming\nProgramming means giving a computer a set of instructions to perform a specific task.\nIt’s like teaching a computer to solve a problem step by step.\nJust as we use English or French to talk to people, programmers use languages like Python to communicate with computers.\n💡 Why Python?\nPython is a beginner-friendly programming language that is:\n* Simple and readable — it looks almost like English.\n\n* Used everywhere — in AI, data science, web development, and automation.\n

In [ ]:
# Loading csv files.
from langchain_community.document_loaders import CSVLoader

loader3 = CSVLoader("data/spreadsheet1.csv")
loaded_data = loader3.load()
loaded_data

[Document(metadata={'source': 'data/spreadsheet1.csv', 'row': 0}, page_content='<!DOCTYPE html><style nonce="1czpMS0bB74jb05Io6ajpg">body{height:100%;margin:0;width:100%}@media (max-height:350px){.button{font-size:10px}.button-container{margin-top:16px}.button.primary-button: /*# sourceMappingURL=style.css.map */</style><script nonce="kK40ilnt3QdKa4LXR7znUg">\'use strict\';function h(a){var b=0;return function(){return b<a.length?{done:!1\n.button.primary-button:active: None\n.button.primary-button:focus: None\n.button.primary-button:hover{padding:4px 12px}.title-text{font-size:22px;line-height:24px}.subtitle-text{font-size:12px;line-height:18px}}@media (min-height:350px){.button{font-size:14px}.button-container{margin-top:16px}.button.primary-button: login_counter];function m(a\n.button.primary-button:hover{padding:12px 24px}.title-text{font-size:28px;line-height:36px}.subtitle-text{font-size:16px;line-height:24px}}.document-root{display:-webkit-box;display:-webkit-flex;display:-moz-b

## Using our Model to generate output based on our data

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# chat_template = ChatPromptTemplate.from_messages(
#     [
#         {
#             ("human", "Anser this {question}, here is the some extra {context}"),
#         }
#     ]
# )

chat_template = ChatPromptTemplate.from_messages([
    ("human", "Answer this {question}, here is some extra {context}")
])

messages = chat_template.format_messages(
    name="Tesla",
    question="What is CAITCC.",
    context = documents1
)

In [ ]:
# send an invoke.
response = chatModel.invoke(messages)
response.content

'**CAITCC – What It Is**\n\nCAITCC is the *Cameroon AI Training & Technology Center* (often referred to simply as the **CAITCC Vocational Training Center**). It is a dedicated training institute that equips learners with the practical skills needed to thrive in the rapidly growing field of artificial intelligence and technology.\n\n**Key Points**\n\n| Feature | Details |\n|---------|---------|\n| **Full Name** | Cameroon AI Training & Technology Center (CAITCC) |\n| **Mission** | Provide hands‑on, industry‑relevant training that turns students into ready‑to‑work AI professionals. |\n| **Core Programs** | • AI Product Manager<br>• AI Product Marketing Manager<br>• AI Engineer |\n| **Curriculum Structure** | • **Phase\u202f1** – Foundations of programming (Python basics, algorithms, data types, control flow, loops, etc.).<br>• **Phase\u202f2** – Intermediate topics (functions, data structures, mini‑projects).<br>• **Phase\u202f3+** – Advanced AI concepts, project work, and real‑world app

In [ ]:

messages = chat_template.format_messages(
    name="Tesla",
    question="if i want to learn python can i learn it they.",
    context = documents1
)
response = chatModel.invoke(messages)
response.content

'Absolutely—**you can learn Python!**  \nThe document you shared already shows you the exact reasons why Python is a great starting point, and it gives you a clear roadmap for the first phase of learning. Here’s how you can turn that roadmap into a practical plan:\n\n| What you’ll learn | Why it matters | Quick start tip |\n|-------------------|----------------|-----------------|\n| **Programming fundamentals** – variables, data types, operators | These are the building blocks of every program. | Write a “Hello, World!” script and play with simple math. |\n| **Control flow** – `if` statements, loops | They let your code make decisions and repeat tasks. | Try a loop that prints numbers 1‑10 and an `if` that checks if a number is even. |\n| **Functions & modules** | They keep code organized and reusable. | Create a function that adds two numbers and call it from a main block. |\n| **Real‑world projects** – sales totals, grade averages, simple automation | Seeing tangible results keeps mo